# **Modulo 1: Facts vs Dims, SCDs**

In [0]:
-- ejercicio 1.5
-- scd type 1, overwrite con Merge

-- Paso 1: Staging view
CREATE OR REPLACE TEMPORARY VIEW staging_zona AS
SELECT
  partido,
  region,
  'Gran Buenos Aires' as ciudad
FROM bootcamp_de_valentin.gold.dim_zona
WHERE
  region = 'gba zona norte'
  AND ciudad = 'GBA'

In [0]:
-- paso 2: conteo filas antes merge
SELECT COUNT(*) FROM bootcamp_de_valentin.gold.dim_zona

In [0]:
select * from staging_zona

In [0]:

-- paso 3: merge solo update
MERGE INTO bootcamp_de_valentin.gold.dim_zona AS target
USING staging_zona AS source
ON target.partido = source.partido AND target.region = source.region
WHEN MATCHED THEN
  UPDATE SET
    target.ciudad = source.ciudad;

In [0]:
-- paso 4 verificar que no se hayan duplicado valroes
SELECT * FROM bootcamp_de_valentin.gold.dim_zona

In [0]:
-- ejercicio 1.6
-- scd type 2, crear columnas con historizacion

-- paso 1, crear la tabla con las columnas necesarias para SCD2
DROP TABLE IF EXISTS bootcamp_de_valentin.gold.dim_zona_scd2;
CREATE TABLE bootcamp_de_valentin.gold.dim_zona_scd2 (
  zona_id       BIGINT GENERATED ALWAYS AS IDENTITY (START WITH 1 INCREMENT BY 1),
  partido       STRING NOT NULL,
  region        STRING NOT NULL,
  ciudad        STRING,
  provincia     STRING DEFAULT 'Buenos Aires',
  pais          STRING DEFAULT 'Argentina',
  valid_from    TIMESTAMP NOT NULL DEFAULT CURRENT_TIMESTAMP(),
  valid_to      TIMESTAMP DEFAULT '9999-12-31',
  is_current    BOOLEAN DEFAULT TRUE
)
USING DELTA
TBLPROPERTIES('delta.feature.allowColumnDefaults' = 'supported')
COMMENT 'Tabla dimensional Zona SCD type 2 - Gold'

In [0]:
-- paso 2, carga inicial desde Silver
INSERT INTO bootcamp_de_valentin.gold.dim_zona_scd2 (partido, region, ciudad)
SELECT DISTINCT
  partido,
  region,
  CASE WHEN region = 'capital federal' THEN 'CABA' ELSE 'GBA' END AS ciudad
FROM bootcamp_de_valentin.silver.propiedades_silver
WHERE
  partido IS NOT NULL

In [0]:
-- verificacion
SELECT * FROM bootcamp_de_valentin.gold.dim_zona_scd2

In [0]:
-- ejercicio 1.7
-- scd2, historizar un cambio manual

-- paso 1, update
UPDATE bootcamp_de_valentin.gold.dim_zona_scd2
SET
  valid_to = current_timestamp(),
  is_current = FALSE
WHERE
  partido = 'capital federal'
  AND is_current = TRUE

In [0]:
-- paso 2, insertar
INSERT INTO bootcamp_de_valentin.gold.dim_zona_scd2 (partido, region, ciudad)
VALUES('capital federal', 'capital federal', 'Buenos Aires')

In [0]:
-- verificacion
SELECT * FROM bootcamp_de_valentin.gold.dim_zona_scd2 ORDER BY partido

In [0]:
-- ejercicio 1.8
-- scd2, historizar con merge masivo
DROP VIEW staging_zona_2;
CREATE TEMPORARY VIEW staging_zona_2 AS
SELECT * FROM VALUES
  ('palermo', 'capital federal', 'CABA Sur'),
  ('barrio-nuevo', 'capital federal', 'CABA'),
  ('capital federal', 'capital federal', 'Ciudad Autonoma de Buenos Aires')
AS t(partido, region, ciudad);

In [0]:
MERGE INTO bootcamp_de_valentin.gold.dim_zona_scd2 AS target
USING staging_zona_2 AS SOURCE
  ON target.partido = source.partido  AND target.region = source.region AND is_current = TRUE
WHEN MATCHED AND target.ciudad <> source.ciudad THEN
  UPDATE SET
  valid_to = current_timestamp(),
  is_current = false
WHEN NOT MATCHED THEN
  INSERT (partido, region, ciudad)
  VALUES (source.partido, source.region, source.ciudad)

In [0]:
-- verificacion post merge
select * from bootcamp_de_valentin.gold.dim_zona_scd2 order by partido

In [0]:
-- paso 3, insertar nuevas versiones para las filas que se cerraron
INSERT INTO bootcamp_de_valentin.gold.dim_zona_scd2 (partido, region, ciudad)
SELECT
  source.partido, source.region, source.ciudad
FROM staging_zona_2 source
JOIN bootcamp_de_valentin.gold.dim_zona_scd2 target ON
  target.partido = source.partido
  AND target.region = source.region
  AND target.is_current = FALSE
  AND target.valid_to >= CURRENT_TIMESTAMP() - INTERVAL 1 MINUTE -- para solamente insertar sobre aquellos registros que se acaban de cerrar y no sobre todos los cerrados
WHERE NOT EXISTS ( -- para asegurarse de que no existe ningun registro actual idem al que se va a insertar
  SELECT 1
  FROM bootcamp_de_valentin.gold.dim_zona_scd2 target2
  WHERE
    target2.partido = source.partido
    AND target2.region = source.region
    AND target2.is_current = TRUE
)

In [0]:
-- verificacion post insert
select * from bootcamp_de_valentin.gold.dim_zona_scd2 order by partido

# **Modulo 2: Dimensiones + Fact, DDL + ETLt**

# **Modulo 3: Queries Analiticas**